# Experiment 4 -- 4-bit QAT Fine-tuning

**Paper Table I, Exp 4**: Exp. 3 + 4-bit QAT (10K fine-tune).  
**Baseline**: Exp 3.  **Metric**: FID, precision.

**Key question**: *Does 4-bit QAT preserve generation quality?*

## Two-stage QAT strategy (Jacob 2018 + Ning 2024)

MZI phase shifters have only 4-6 bit effective precision (Ning 2024).  
Naive rounding of float32 weights to 4 bits causes -11% to -14% accuracy  
drop (Jacob 2018, Table 4.7). QAT solves this by simulating quantization  
during training so the model learns to be robust to limited precision.

```
Stage 1 (exp3):  Float32 + photonic noise training -> converged checkpoint
Stage 2 (THIS):  Load exp3 checkpoint -> wrap with QATWrapper(bits=4)
                 -> fine-tune 10K steps with lower lr (1e-5)
```

## How QAT works (Jacob 2018, Section 3)

```
Forward:   float_weight -> clamp -> scale -> round -> descale -> fake_quantized
Backward:  gradient passes through round() as identity (STE)
Optimizer: updates the float32 weights using gradients from quantized forward
```

The STE (straight-through estimator) is the key trick: round() has zero  
gradient, but STE pretends it's the identity. Gradients computed at the  
quantized value are applied to the float weight.

## Why 4 bits?

Ning 2024: MZI phase shifters have 4-6 bit effective precision due to  
DAC resolution, thermal drift, and manufacturing variation.  
4 bits = 16 discrete phase levels = conservative floor.

**Sources:**
- Jacob et al. 2018 -- QAT framework, STE, r=S(q-Z) (CVPR)
- Ning et al. 2024 -- MZI 4-6 bit precision (J. Lightwave Technol.)
- Ning et al. 2025 -- StrC-ONN validates noise+quant training (Optica)
- PhotonFlow paper draft -- Table I exp4, Stage 3 bottom panel

In [ ]:
# -- 1. Environment detection + dependency install -------------------------
import sys, subprocess, os

IN_COLAB  = False
IN_KAGGLE = False
IN_LOCAL  = False

if os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    IN_KAGGLE = True
    ENV_NAME  = "Kaggle"
else:
    try:
        import google.colab
        IN_COLAB = True
        ENV_NAME = "Colab"
    except ImportError:
        IN_LOCAL = True
        ENV_NAME = "Local"

print(f"Environment: {ENV_NAME}")

def _pip_install(*pkgs):
    for pkg in pkgs:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", pkg],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
            print(f"  [ok] {pkg}")
        except subprocess.CalledProcessError:
            print(f"  [warn] {pkg} -- install failed")
            if IN_KAGGLE:
                print("         Kaggle: Settings > Internet > toggle ON")

if IN_COLAB or IN_KAGGLE:
    print("Installing dependencies...")
    _pip_install("tqdm", "pyyaml", "wandb")

import torch
assert torch.cuda.is_available(), "GPU not found"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# -- 2. Repo / sys.path setup ----------------------------------------------
REPO_URL    = "https://github.com/HasinthakaPiyumal/photon-flow-research.git"
REPO_BRANCH = "h/phase1"

if IN_COLAB:
    REPO_DIR = "/content/photon-flow-research"
elif IN_KAGGLE:
    REPO_DIR = "/kaggle/working/photon-flow-research"
else:
    REPO_DIR = "."

if (IN_COLAB or IN_KAGGLE) and not os.path.exists(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR
    ])
    print(f"Cloned repo (branch={REPO_BRANCH}) to {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo  : {os.path.abspath(REPO_DIR)}")
print(f"Branch: {REPO_BRANCH}")

In [ ]:
# -- 3. Imports -------------------------------------------------------------
import math, json
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# PhotonFlow core
from photonflow.model import PhotonFlowModel
from photonflow.train import CFMLoss, euler_sample

# QAT (Jacob 2018)
from hardware.qat import QATWrapper, fake_quantize

# Evaluation
from eval.fid import FIDCalculator
from eval.metrics import compute_precision_recall

device = torch.device("cuda")
print(f"Device: {device}")
print("Imports OK -- PhotonFlowModel + QATWrapper + FIDCalculator loaded")

In [ ]:
# -- W&B init --------------------------------------------------------------
import wandb

WANDB_API_KEY = None
if IN_COLAB:
    try:
        from google.colab import userdata
        WANDB_API_KEY = userdata.get("WANDB_API_KEY")
    except Exception: pass
elif IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception: pass
else:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print("W&B login: OK")
else:
    print("[warn] WANDB_API_KEY not found -- offline mode")
    os.environ["WANDB_MODE"] = "offline"

wandb_run = None
print("W&B ready")

In [ ]:
# -- 4. Experiment config + W&B run init -----------------------------------
import yaml

config_path = os.path.join(REPO_DIR, "configs", "exp4_qat.yaml")
with open(config_path) as f:
    yaml_cfg = yaml.safe_load(f)

print(f"Loaded config: {config_path}")

CFG = {
    "in_dim"     : yaml_cfg["model"]["in_dim"],
    "hidden_dim" : yaml_cfg["model"]["hidden_dim"],
    "num_blocks" : yaml_cfg["model"]["num_blocks"],
    "use_noise"  : yaml_cfg["model"]["use_noise"],
    "sigma_s"    : yaml_cfg["model"]["sigma_s"],
    "sigma_t"    : yaml_cfg["model"]["sigma_t"],
    "dataset"    : yaml_cfg["data"]["dataset"],
    "batch_size" : yaml_cfg["data"]["batch_size"],
    "data_root"  : os.path.join(REPO_DIR, yaml_cfg["data"]["root"]),
    "lr"               : yaml_cfg["training"]["lr"],
    "total_steps"      : yaml_cfg["training"]["total_steps"],
    "checkpoint_every" : yaml_cfg["training"]["checkpoint_every"],
    "sample_every"     : yaml_cfg["training"]["sample_every"],
    "sample_steps"     : yaml_cfg["training"]["sample_steps"],
    "seed"             : yaml_cfg["training"]["seed"],
    "qat_enabled" : yaml_cfg["qat"]["enabled"],
    "qat_bits"    : yaml_cfg["qat"]["bits"],
    "qat_checkpoint" : os.path.join(REPO_DIR, yaml_cfg["qat"]["checkpoint"]),
    "output_dir" : os.path.join(REPO_DIR, yaml_cfg["output_dir"]),
}
CFG["time_dim"] = yaml_cfg["model"].get("time_dim", CFG["hidden_dim"])

IN_DIM = CFG["in_dim"]

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])

OUTPUT_DIR = CFG["output_dir"]
CKPT_DIR   = os.path.join(OUTPUT_DIR, "checkpoints")
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
for d in [CKPT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

wandb_run = wandb.init(
    project="photonflow",
    name=f"exp4-qat-{CFG['qat_bits']}bit-{CFG['total_steps']//1000}K",
    config=CFG,
    tags=["exp4", "qat", f"{CFG['qat_bits']}bit", "mnist", ENV_NAME.lower()],
)

print(f"QAT: {CFG['qat_bits']}-bit, lr={CFG['lr']}, {CFG['total_steps']:,} steps")
print(f"Load exp3 checkpoint: {CFG['qat_checkpoint']}")
print(f"W&B run: {wandb_run.url}")

In [ ]:
# -- 5. MNIST DataLoader ---------------------------------------------------
_tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda img: img.view(-1)),
])

train_ds = datasets.MNIST(CFG["data_root"], train=True,  download=True, transform=_tfm)
test_ds  = datasets.MNIST(CFG["data_root"], train=False, download=True, transform=_tfm)

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"],
    shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)

print(f"Train: {len(train_ds):,}  Test: {len(test_ds):,}")
print(f"Batch size: {CFG['batch_size']}")

In [ ]:
# -- 6. Load exp3 checkpoint + wrap with QATWrapper -------------------------
# Two-stage strategy (Jacob 2018):
#   Stage 1: exp3 trained with noise, no quantization -> converged weights
#   Stage 2: THIS -- load those weights, wrap with QAT, fine-tune

# Build base model
base_model = PhotonFlowModel(
    in_dim     = CFG["in_dim"],
    hidden_dim = CFG["hidden_dim"],
    num_blocks = CFG["num_blocks"],
    time_dim   = CFG["time_dim"],
    use_noise  = CFG["use_noise"],
    sigma_s    = CFG["sigma_s"],
    sigma_t    = CFG["sigma_t"],
).to(device)

# Load exp3 checkpoint
ckpt_path = CFG["qat_checkpoint"]
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    base_model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded exp3 checkpoint: {ckpt_path}")
    print(f"  trained step: {ckpt.get('step', '?')}")
else:
    print(f"[WARN] Checkpoint not found: {ckpt_path}")
    print("  Training from scratch (exp3 checkpoint required for proper QAT)")
    print("  Run notebook 04 (exp3) first to generate the checkpoint.")

# Wrap with QATWrapper (Jacob 2018: insert fake-quantize in forward pass)
# During training: weights are fake-quantized to 4-bit (16 levels)
# During eval: QAT is bypassed (uses float weights)
model = QATWrapper(base_model, bits=CFG["qat_bits"], enabled=True)
model.to(device)

# Lower lr for fine-tuning (Jacob 2018: QAT uses lower lr than initial training)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
loss_fn   = CFMLoss(sigma_min=0.0)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel: PhotonFlowModel + QATWrapper({CFG['qat_bits']}-bit)")
print(f"  params     = {total_params:,}")
print(f"  lr         = {CFG['lr']} (10x lower than exp3)")
print(f"  noise      = {CFG['use_noise']}")
print(f"  QAT levels = {2**CFG['qat_bits']} ({CFG['qat_bits']}-bit)")

wandb.log({"model/total_params": total_params, "qat/bits": CFG["qat_bits"]})

In [ ]:
# -- 7. QAT fine-tuning loop (10K steps) -----------------------------------
# Jacob 2018 Algorithm 1:
#   1. Insert fake-quantize nodes (QATWrapper does this)
#   2. Train with STE backward (FakeQuantize handles this)
#   3. Float weights are updated by optimizer
#   4. On next forward, weights are re-quantized

import wandb
losses   = []
step_log = []

data_iter = iter(train_loader)
model.train()

pbar = tqdm(range(CFG["total_steps"]), desc="exp4 QAT", dynamic_ncols=True)

for step in pbar:

    try:
        x1, _ = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        x1, _ = next(data_iter)

    x1 = x1.to(device)

    loss = loss_fn(model, x1)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    losses.append(loss.item())
    wandb.log({"train/loss": loss.item()}, step=step)

    if step % 100 == 0:
        avg = float(np.mean(losses[max(0, len(losses)-100):]))
        step_log.append((step, avg))
        pbar.set_postfix(loss=f"{avg:.4f}")
        wandb.log({"train/loss_avg100": avg}, step=step)

    if (step + 1) % CFG["checkpoint_every"] == 0:
        ckpt_path = os.path.join(CKPT_DIR, f"step_{step+1:07d}.pt")
        torch.save({
            "step": step + 1,
            "model_state_dict": base_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "losses": losses,
            "config": CFG,
        }, ckpt_path)
        art = wandb.Artifact(f"exp4-qat-step{step+1}", type="model")
        art.add_file(ckpt_path)
        wandb.log_artifact(art)
        print(f"\n[step {step+1:,}] Checkpoint -> {ckpt_path}")

    if (step + 1) % CFG["sample_every"] == 0:
        base_model.eval()
        with torch.no_grad():
            samp = euler_sample(base_model, shape=(64, IN_DIM),
                                num_steps=CFG["sample_steps"], device=device)
        samp = samp.clamp(0.0, 1.0).cpu().view(64, 1, 28, 28)

        fig, axes = plt.subplots(8, 8, figsize=(8, 8))
        for i, ax in enumerate(axes.flat):
            ax.imshow(samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
        avg_now = float(np.mean(losses[max(0, len(losses)-500):]))
        fig.suptitle(f"Exp4 QAT  step={step+1:,}  loss={avg_now:.4f}", fontsize=11)
        plt.tight_layout()
        fig_path = os.path.join(FIG_DIR, f"samples_step{step+1:07d}.png")
        plt.savefig(fig_path, dpi=100, bbox_inches="tight")
        wandb.log({f"samples/step_{step+1}": wandb.Image(fig)}, step=step)
        plt.show()
        plt.close()
        base_model.train()

pbar.close()
print(f"\nQAT fine-tuning complete: {step+1:,} steps, final loss: {losses[-1]:.4f}")

In [ ]:
# -- 8. Loss curve ---------------------------------------------------------
import wandb
steps_arr  = [s for s, _ in step_log]
losses_arr = [l for _, l in step_log]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(steps_arr, losses_arr, lw=1.5, alpha=0.85, color="crimson")
ax.set_xlabel("Fine-tuning step", fontsize=12)
ax.set_ylabel("CFM loss (MSE)", fontsize=12)
ax.set_title(
    f"Exp4: {CFG['qat_bits']}-bit QAT Fine-tuning ({CFG['total_steps']//1000}K steps)",
    fontsize=13
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
curve_path = os.path.join(FIG_DIR, "loss_curve.png")
plt.savefig(curve_path, dpi=150)
wandb.log({"charts/loss_curve": wandb.Image(fig)})
plt.show()
print(f"Loss curve saved: {curve_path}")

In [ ]:
# -- 9. Final sample grid --------------------------------------------------
import wandb
base_model.eval()
with torch.no_grad():
    final_samp = euler_sample(base_model, shape=(100, IN_DIM),
                              num_steps=CFG["sample_steps"], device=device)
final_samp = final_samp.clamp(0.0, 1.0).cpu().view(100, 1, 28, 28)

fig, axes = plt.subplots(10, 10, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(final_samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle(f"Exp4 Final Samples -- {CFG['qat_bits']}-bit QAT", fontsize=13)
plt.tight_layout()
samp_path = os.path.join(FIG_DIR, "final_samples.png")
plt.savefig(samp_path, dpi=150, bbox_inches="tight")
wandb.log({"samples/final_10x10": wandb.Image(fig)})
plt.show()

In [ ]:
# -- 10. FID + Precision/Recall --------------------------------------------
# Paper Table I exp4 metrics: FID, precision
import wandb

N_FID    = 10_000
fid_calc = FIDCalculator(device=device)

# Real images
real_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)
real_imgs = []
for imgs, _ in real_loader:
    real_imgs.append(imgs.view(-1, 1, 28, 28))
real_imgs = torch.cat(real_imgs, dim=0)[:N_FID]
print(f"Real images: {real_imgs.shape}")

# Generated images (using base_model directly, not QAT wrapper)
base_model.eval()
gen_batches = []
n_generated = 0
with torch.no_grad():
    pbar_fid = tqdm(total=N_FID, desc="Generating for FID")
    while n_generated < N_FID:
        bs = min(256, N_FID - n_generated)
        samp = euler_sample(base_model, shape=(bs, IN_DIM),
                            num_steps=CFG["sample_steps"], device=device)
        gen_batches.append(samp.clamp(0.0, 1.0).cpu().view(bs, 1, 28, 28))
        n_generated += bs
        pbar_fid.update(bs)
    pbar_fid.close()
gen_imgs = torch.cat(gen_batches, dim=0)[:N_FID]

# FID
print("Extracting features...")
real_feats = fid_calc.extract_features(real_imgs, batch_size=256)
gen_feats  = fid_calc.extract_features(gen_imgs, batch_size=256)
real_stats = fid_calc.compute_statistics(real_feats)
gen_stats  = fid_calc.compute_statistics(gen_feats)
fid_score  = fid_calc.compute_fid(real_stats, gen_stats)

print(f"\nFID (exp4, {CFG['qat_bits']}-bit QAT): {fid_score:.2f}")
wandb.log({"eval/fid": fid_score})

# Precision/Recall (Kynkaanniemi 2019)
# Use a subset for memory (P/R needs N^2 distance matrix)
N_PR = min(2000, N_FID)
precision, recall = compute_precision_recall(
    real_feats[:N_PR], gen_feats[:N_PR], k=3
)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
wandb.log({"eval/precision": precision, "eval/recall": recall})

# Compare against exp3
exp3_path = os.path.join(REPO_DIR, "outputs", "exp3_noise_regularized", "results_exp3.json")
if os.path.exists(exp3_path):
    with open(exp3_path) as f:
        exp3_results = json.load(f)
    exp3_fid = exp3_results["fid"]
    delta = ((fid_score - exp3_fid) / exp3_fid) * 100
    print(f"\nExp3 FID (pre-QAT): {exp3_fid:.2f}")
    print(f"QAT impact: {delta:+.1f}%")
    wandb.log({"eval/exp3_fid": exp3_fid, "eval/fid_delta_pct": delta})
else:
    print("(exp3 results not found -- run notebook 04 first)")

In [ ]:
# -- 11. Results summary + W&B finish --------------------------------------
import json as _json, wandb

results = {
    "experiment"   : "exp4_qat_finetune",
    "environment"  : ENV_NAME,
    "description"  : f"PhotonFlow + noise + {CFG['qat_bits']}-bit QAT fine-tuning",
    "total_steps"  : CFG["total_steps"],
    "fid"          : round(float(fid_score), 4),
    "precision"    : round(float(precision), 4),
    "recall"       : round(float(recall), 4),
    "final_loss_avg1000": round(float(np.mean(losses[-min(1000,len(losses)):])), 6),
    "model_params" : total_params,
    "qat"          : {
        "bits"       : CFG["qat_bits"],
        "levels"     : 2 ** CFG["qat_bits"],
        "lr"         : CFG["lr"],
        "source_ckpt": CFG["qat_checkpoint"],
    },
    "sources" : [
        "Jacob et al. 2018 -- QAT with STE (CVPR, Section 3)",
        "Ning et al. 2024 -- MZI 4-6 bit precision (J. Lightwave Technol.)",
        "Ning et al. 2025 -- StrC-ONN noise+quant training (Optica)",
        "Kynkaanniemi et al. 2019 -- Precision/Recall metric (NeurIPS)",
    ],
}

results_path = os.path.join(OUTPUT_DIR, "results_exp4.json")
with open(results_path, "w") as f:
    _json.dump(results, f, indent=2)

np.save(os.path.join(OUTPUT_DIR, "losses_exp4.npy"), np.array(losses))

final_ckpt = os.path.join(CKPT_DIR, "exp4_final.pt")
torch.save({
    "step": CFG["total_steps"],
    "model_state_dict": base_model.state_dict(),
    "fid": fid_score,
    "precision": precision,
    "recall": recall,
    "config": CFG,
}, final_ckpt)

wandb.log({
    "summary/fid"       : fid_score,
    "summary/precision" : precision,
    "summary/recall"    : recall,
})
wandb.finish()

SEP = "=" * 58
print(SEP)
print("EXP4 QAT RESULTS")
print(SEP)
print(f"  FID:              {fid_score:.2f}")
print(f"  Precision:        {precision:.4f}")
print(f"  Recall:           {recall:.4f}")
print(f"  QAT bits:         {CFG['qat_bits']}")
print(f"  Fine-tune steps:  {CFG['total_steps']:,}")
print(f"  Fine-tune lr:     {CFG['lr']}")
print(f"  Environment:      {ENV_NAME}")
print(SEP)
print(f"  Results    -> {results_path}")
print(f"  Checkpoint -> {final_ckpt}")
print(SEP)
print()
print("Next: notebooks/06_exp6_hardware_simulation.ipynb")
print("      Load exp4 checkpoint -> MZI profiler -> photonic metrics")